# Notebook 2 — Silver Layer Transformation

**Project:** Tractor Industry Analytics  
**Layer:** Silver (clean, standardise, enrich → curated Delta tables)  
**Author:** Tractor Analytics Pipeline  
**Last Updated:** 2026-06-18

## Overview
This notebook reads every Bronze table and produces cleaned, enriched Silver tables
ready for analytical aggregation in Notebook 3 (Gold).

### Transformations applied per table
| Bronze Source | Silver Target | Key Transformations |
|---|---|---|
| `bronze.india_tractor_sales` | `silver.india_tractor_sales` | Calendar year column, YoY growth %, 3-year rolling avg |
| `bronze.india_tractor_seasonality` | `silver.india_tractor_seasonality` | Month order corrected, season_group bucket added |
| `bronze.india_tractor_brands` | `silver.india_tractor_brands` | Rank column, share normalised, HP range parsed |
| `bronze.us_aem_tractor_sales_hp` | `silver.us_aem_tractor_sales_hp` | HP share %, YoY growth, total reconciliation |
| `bronze.global_tractor_production` | `silver.global_tractor_production` | Export intensity %, domestic retention %, standardised country names |
| `bronze.comtrade_hs8701_exports` | `silver.comtrade_hs8701_exports` | M49 code → country name, value per unit, flow label |
| `bronze.fred_*` (5 series) | `silver.fred_annual_indices` | Annual averages unified into one wide table |
| `bronze.faostat_arable_land` | `silver.faostat_arable_land` | Key-country filter, standardised area name |
| *(cross-country join)* | `silver.global_tractor_sales` | Unified schema: `year`, `country`, `hp_category`, `units_sold` |

## Prerequisites
- **Notebook 01** (`01_bronze_ingestion.ipynb`) must have been run successfully.
- All `bronze.*` tables must exist in the metastore.
- Cluster: Databricks Runtime 13.x+ (Delta Lake 2.x, PySpark 3.4+).

## Cell 1 — Environment Setup

In [ ]:
import os
from datetime import datetime, timezone

# ── Runtime detection ───────────────────────────────────────────────────────
try:
    dbutils  # noqa: F821
    ON_DATABRICKS = True
except NameError:
    ON_DATABRICKS = False

print(f"Running on Databricks: {ON_DATABRICKS}")

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
TRANSFORMED_AT = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"Transform timestamp : {TRANSFORMED_AT}")

## Cell 2 — Spark Session & Silver Schema Bootstrap

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType, DateType, BooleanType
)

if ON_DATABRICKS:
    spark = SparkSession.builder.getOrCreate()
else:
    spark = (
        SparkSession.builder
        .appName("silver_transformation_local")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

spark.conf.set("spark.sql.shuffle.partitions", "8")

# Create silver schema (idempotent)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"USE {SILVER_SCHEMA}")
print(f"Active schema: {SILVER_SCHEMA}")

## Cell 3 — Helper: Write Silver Table

Appends standard silver-layer lineage columns and writes as Delta (overwrite = idempotent).

In [ ]:
def write_silver(df, table_name: str, description: str = "") -> None:
    """
    Append silver lineage columns and write as a Delta table.

    Parameters
    ----------
    df         : transformed Spark DataFrame
    table_name : target table name (without schema prefix)
    description: short description for log output
    """
    full_table = f"{SILVER_SCHEMA}.{table_name}"

    df = (
        df
        .withColumn("_transformed_at",   F.lit(TRANSFORMED_AT))
        .withColumn("_silver_layer",      F.lit(True))
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )

    count = spark.table(full_table).count()
    label = description or table_name
    print(f"[OK] {full_table:<50s}  {count:>7,} rows   {label}")

## Cell 4 — `silver.india_tractor_sales`

**Source:** `bronze.india_tractor_sales`  
**Transformations:**
- Add `calendar_year` = `year_end` (Indian FY runs April–March, so FY 2023-24 maps to CY 2024 as the dominant calendar year)
- Compute `yoy_growth_pct` — year-over-year growth percentage
- Compute `rolling_avg_3yr` — 3-year centred rolling average to smooth cyclicality
- Add `is_provisional` flag from the source string
- Drop internal bronze metadata columns

In [ ]:
raw = spark.table(f"{BRONZE_SCHEMA}.india_tractor_sales")

# ── Window for lag / rolling average ────────────────────────────────────────
w_order   = Window.orderBy("year_start")
w_rolling = Window.orderBy("year_start").rowsBetween(-1, 1)   # 3-year centred window

india_sales = (
    raw
    # calendar year = the year in which the fiscal year ends (dominant half)
    .withColumn("calendar_year",  F.col("year_end"))
    # provisional flag — true if source string contains 'Provisional'
    .withColumn("is_provisional",
                F.col("source").contains("Provisional") | F.col("source").contains("provisional"))
    # YoY growth %
    .withColumn("prev_units",      F.lag("total_units_sold", 1).over(w_order))
    .withColumn("yoy_growth_pct",
                F.round(
                    (F.col("total_units_sold") - F.col("prev_units")) / F.col("prev_units") * 100,
                    2
                )
    )
    # 3-year rolling average (centred)
    .withColumn("rolling_avg_3yr",
                F.round(F.avg("total_units_sold").over(w_rolling), 0))
    # clean up intermediary and bronze metadata columns
    .drop("prev_units", "_source_file", "_ingested_at", "_bronze_layer")
    # final column order
    .select(
        "fiscal_year", "year_start", "year_end", "calendar_year",
        "total_units_sold", "yoy_growth_pct", "rolling_avg_3yr",
        "is_provisional", "source"
    )
)

write_silver(india_sales, "india_tractor_sales", "India annual sales with growth metrics")

spark.table("silver.india_tractor_sales") \
    .select("fiscal_year", "calendar_year", "total_units_sold", "yoy_growth_pct", "rolling_avg_3yr") \
    .orderBy("year_start") \
    .show(truncate=False)

## Cell 5 — `silver.india_tractor_seasonality`

**Source:** `bronze.india_tractor_seasonality`  
**Transformations:**
- Indian fiscal year starts in April, so fiscal month 1 = April (calendar month 4). Add `calendar_month` mapping.
- Add `season_group`: HIGH (Jun–Aug), MEDIUM (Apr–May, Sep–Nov, Mar), LOW (Dec–Feb)
- Verify shares sum to 100 % and flag any discrepancy

In [ ]:
raw_season = spark.table(f"{BRONZE_SCHEMA}.india_tractor_seasonality")

# Fiscal month 1 = April → calendar month 4; add 3, wrap with modulo
# fiscal 1→4, 2→5, ..., 9→12, 10→1, 11→2, 12→3
seasonality = (
    raw_season
    .withColumn("calendar_month",
                F.expr("((month + 2) % 12) + 1"))     # (1+2)%12+1=4, (9+2)%12+1=12, (10+2)%12+1=1
    .withColumn("season_group",
                F.when(F.col("calendar_month").isin(6, 7, 8), "HIGH")
                 .when(F.col("calendar_month").isin(12, 1, 2),  "LOW")
                 .otherwise("MEDIUM"))
    # cumulative share for ordering in charts
    .withColumn("cumulative_share_pct",
                F.round(
                    F.sum("avg_share_pct").over(Window.orderBy("month")
                                                      .rowsBetween(Window.unboundedPreceding, 0)),
                    2
                ))
    .drop("_source_file", "_ingested_at", "_bronze_layer")
    .select(
        "month", "calendar_month", "month_name",
        "avg_share_pct", "cumulative_share_pct",
        "season_label", "season_group", "notes"
    )
)

write_silver(seasonality, "india_tractor_seasonality", "India monthly seasonality with groups")

# Validate: shares must sum to ~100
total_share = spark.table("silver.india_tractor_seasonality").agg(F.sum("avg_share_pct")).collect()[0][0]
print(f"\nTotal share sum = {total_share:.1f}%  ({'OK' if abs(total_share - 100) < 0.5 else 'WARNING: does not sum to 100'})")

spark.table("silver.india_tractor_seasonality") \
    .select("month", "calendar_month", "month_name", "avg_share_pct", "season_group", "season_label") \
    .orderBy("month") \
    .show(truncate=False)

## Cell 6 — `silver.india_tractor_brands`

**Source:** `bronze.india_tractor_brands`  
**Transformations:**
- Add `market_share_rank` (1 = largest)
- Normalise shares so they sum to exactly 100 % (compensate for rounding in source)
- Extract `hp_min` / `hp_max` from the free-text `hp_range_focus` field
- Add `is_indian_brand` flag

In [ ]:
raw_brands = spark.table(f"{BRONZE_SCHEMA}.india_tractor_brands")

# Total share for normalisation
total_share_val = raw_brands.agg(F.sum("market_share_2023_pct")).collect()[0][0]

brands = (
    raw_brands
    # rank by descending share
    .withColumn("market_share_rank",
                F.rank().over(Window.orderBy(F.desc("market_share_2023_pct"))))
    # normalise to sum-to-100
    .withColumn("market_share_2023_pct_norm",
                F.round(F.col("market_share_2023_pct") / F.lit(total_share_val) * 100, 2))
    # extract HP min/max from strings like '20-75 HP'
    .withColumn("hp_min",
                F.regexp_extract("hp_range_focus", r"(\d+)\s*-\s*\d+", 1).cast(IntegerType()))
    .withColumn("hp_max",
                F.regexp_extract("hp_range_focus", r"\d+\s*-\s*(\d+)", 1).cast(IntegerType()))
    # domestic vs foreign brand flag
    .withColumn("is_indian_brand",
                F.col("country_of_origin") == "India")
    .drop("_source_file", "_ingested_at", "_bronze_layer")
    .select(
        "market_share_rank", "brand", "parent_company", "country_of_origin",
        "is_indian_brand", "market_share_2023_pct", "market_share_2023_pct_norm",
        "hp_range_focus", "hp_min", "hp_max", "key_models", "notes"
    )
)

write_silver(brands, "india_tractor_brands", "India brand market share with rank and HP parsed")

spark.table("silver.india_tractor_brands") \
    .select("market_share_rank", "brand", "country_of_origin",
            "market_share_2023_pct", "market_share_2023_pct_norm", "hp_min", "hp_max") \
    .orderBy("market_share_rank") \
    .show(truncate=False)

## Cell 7 — `silver.us_aem_tractor_sales_hp`

**Source:** `bronze.us_aem_tractor_sales_hp`  
**Transformations:**
- Rename column `40_to_100hp_units` to `hp40_to_100_units` (avoid backtick requirement)
- Compute HP segment share percentages
- Compute YoY growth per segment
- Add `dominant_segment` = segment with highest share that year

In [ ]:
raw_us = spark.table(f"{BRONZE_SCHEMA}.us_aem_tractor_sales_hp")

w_yr = Window.orderBy("year")

us_hp = (
    raw_us
    # rename awkward column names
    .withColumnRenamed("40_to_100hp_units", "hp40_to_100_units")
    .withColumnRenamed("4wd_units",          "hp_4wd_units")
    # use total_2wd_4wd as denominator; if null fall back to sum of three segments
    .withColumn("total_units",
                F.coalesce(
                    F.col("total_2wd_4wd"),
                    F.col("under_40hp_units") + F.col("hp40_to_100_units") + F.col("over_100hp_units")
                ))
    # HP share percentages
    .withColumn("under_40hp_share_pct",
                F.round(F.col("under_40hp_units")  / F.col("total_units") * 100, 1))
    .withColumn("hp40_to_100_share_pct",
                F.round(F.col("hp40_to_100_units") / F.col("total_units") * 100, 1))
    .withColumn("over_100hp_share_pct",
                F.round(F.col("over_100hp_units")  / F.col("total_units") * 100, 1))
    # YoY growth on total units
    .withColumn("prev_total",    F.lag("total_units", 1).over(w_yr))
    .withColumn("yoy_growth_pct",
                F.round((F.col("total_units") - F.col("prev_total")) / F.col("prev_total") * 100, 2))
    # dominant segment by unit volume
    .withColumn("dominant_segment",
                F.when(
                    (F.col("under_40hp_units") >= F.col("hp40_to_100_units")) &
                    (F.col("under_40hp_units") >= F.col("over_100hp_units")),
                    "<40HP"
                ).when(
                    F.col("hp40_to_100_units") >= F.col("over_100hp_units"),
                    "40-100HP"
                ).otherwise("100+HP"))
    # provisional flag
    .withColumn("is_provisional",
                F.col("source").contains("Provisional") | F.col("source").contains("YTD"))
    .drop("prev_total", "_source_file", "_ingested_at", "_bronze_layer")
    .select(
        "year",
        "under_40hp_units",  "hp40_to_100_units",  "over_100hp_units",  "hp_4wd_units",
        "total_units",
        "under_40hp_share_pct", "hp40_to_100_share_pct", "over_100hp_share_pct",
        "yoy_growth_pct", "dominant_segment", "is_provisional", "source"
    )
)

write_silver(us_hp, "us_aem_tractor_sales_hp", "US AEM sales with HP share % and YoY growth")

spark.table("silver.us_aem_tractor_sales_hp") \
    .select("year", "total_units", "under_40hp_share_pct", "hp40_to_100_share_pct",
            "over_100hp_share_pct", "yoy_growth_pct", "dominant_segment") \
    .orderBy("year") \
    .show(truncate=False)

## Cell 8 — `silver.global_tractor_production`

**Source:** `bronze.global_tractor_production`  
**Transformations:**
- Standardise country names to a canonical set used throughout Silver/Gold layers
- Compute `export_intensity_pct` = exports / production × 100
- Compute `domestic_retention_pct` = domestic_sales / production × 100
- Add `production_rank` per year

In [ ]:
raw_prod = spark.table(f"{BRONZE_SCHEMA}.global_tractor_production")

# Canonical country name mapping (ISO2 → display name)
country_name_map = {
    "IN": "India",
    "CN": "China",
    "US": "United States",
    "DE": "Germany",
    "JP": "Japan",
    "IT": "Italy",
    "FR": "France",
    "GB": "United Kingdom",
    "BR": "Brazil",
    "TR": "Turkey",
}

# Build a mapping expression from the dict
country_map_expr = F.create_map([F.lit(v) for pair in country_name_map.items() for v in pair])

w_yr_prod = Window.partitionBy("year").orderBy(F.desc("tractors_produced_units"))

global_prod = (
    raw_prod
    .withColumn("country_name",
                F.coalesce(
                    country_map_expr[F.col("country_code")],
                    F.col("country")    # fall back to original if code not in map
                ))
    .withColumn("export_intensity_pct",
                F.round(
                    F.col("tractors_exported_units").cast("double")
                    / F.col("tractors_produced_units") * 100,
                    1
                ))
    .withColumn("domestic_retention_pct",
                F.round(
                    F.col("domestic_sales_estimate").cast("double")
                    / F.col("tractors_produced_units") * 100,
                    1
                ))
    .withColumn("production_rank",
                F.rank().over(w_yr_prod))
    .drop("_source_file", "_ingested_at", "_bronze_layer")
    .select(
        "year", "country_name", "country_code",
        "tractors_produced_units", "tractors_exported_units", "domestic_sales_estimate",
        "export_intensity_pct", "domestic_retention_pct", "production_rank", "notes"
    )
)

write_silver(global_prod, "global_tractor_production",
             "Global production with export intensity and production rank")

spark.table("silver.global_tractor_production") \
    .select("year", "country_name", "tractors_produced_units",
            "export_intensity_pct", "domestic_retention_pct", "production_rank") \
    .orderBy("year", "production_rank") \
    .show(20, truncate=False)

## Cell 9 — `silver.comtrade_hs8701_exports`

**Source:** `bronze.comtrade_hs8701_exports`  
**Transformations:**
- Map UN M49 numeric country codes → ISO2 code and display name
- Translate `flowCode` ('X'/'M') → readable label ('Export'/'Import')
- Compute `usd_per_unit` = `fobValue_usd` / `qty` (average unit value — proxy for equipment price tier)
- Rename columns to `snake_case`
- Drop duplicate rows (the raw Comtrade data can have duplicate reporter/partner pairs)

In [ ]:
raw_trade = spark.table(f"{BRONZE_SCHEMA}.comtrade_hs8701_exports")

# ── UN M49 → ISO2 + display name lookup ─────────────────────────────────────
# Major tractor-trade countries; extend as needed
M49_TO_ISO2 = {
    36: ("AU", "Australia"),      40: ("AT", "Austria"),
    56: ("BE", "Belgium"),        76: ("BR", "Brazil"),
    124: ("CA", "Canada"),        156: ("CN", "China"),
    203: ("CZ", "Czechia"),       208: ("DK", "Denmark"),
    246: ("FI", "Finland"),       250: ("FR", "France"),
    276: ("DE", "Germany"),       300: ("GR", "Greece"),
    356: ("IN", "India"),         392: ("JP", "Japan"),
    380: ("IT", "Italy"),         410: ("KR", "South Korea"),
    484: ("MX", "Mexico"),        528: ("NL", "Netherlands"),
    616: ("PL", "Poland"),        620: ("PT", "Portugal"),
    643: ("RU", "Russia"),        724: ("ES", "Spain"),
    752: ("SE", "Sweden"),        756: ("CH", "Switzerland"),
    792: ("TR", "Turkey"),        804: ("UA", "Ukraine"),
    826: ("GB", "United Kingdom"), 840: ("US", "United States"),
    858: ("UY", "Uruguay"),       710: ("ZA", "South Africa"),
    0:   ("WLD", "World"),         490: ("TW", "Taiwan"),
    918: ("EU", "European Union"), 97: ("EU", "European Union"),
}

# Create broadcast lookup DataFrames
m49_schema = StructType([
    StructField("m49_code",    IntegerType(), False),
    StructField("iso2_code",   StringType(),  True),
    StructField("country_name", StringType(), True),
])

m49_rows = [(k, v[0], v[1]) for k, v in M49_TO_ISO2.items()]
m49_df   = spark.createDataFrame(m49_rows, schema=m49_schema)

# ── Apply transformations ────────────────────────────────────────────────────
comtrade = (
    raw_trade
    # deduplicate — Comtrade sometimes has duplicate records for the same flow
    .dropDuplicates(["year", "reporterCode", "flowCode", "partnerCode", "fobValue_usd", "qty"])
    # rename to snake_case
    .withColumnRenamed("reporterCode",    "reporter_m49")
    .withColumnRenamed("flowCode",         "flow_code")
    .withColumnRenamed("partnerCode",      "partner_m49")
    .withColumnRenamed("netWgt_kg",        "net_weight_kg")
    .withColumnRenamed("fobValue_usd",     "fob_value_usd")
    .withColumnRenamed("primaryValue_usd", "primary_value_usd")
    .withColumnRenamed("qty",              "quantity_units")
    # human-readable flow label
    .withColumn("flow_label",
                F.when(F.col("flow_code") == "X", "Export")
                 .when(F.col("flow_code") == "M", "Import")
                 .otherwise(F.col("flow_code")))
    # average unit value (USD per tractor) — useful for price tier analysis
    .withColumn("usd_per_unit",
                F.when(F.col("quantity_units") > 0,
                       F.round(F.col("fob_value_usd") / F.col("quantity_units"), 0))
                 .otherwise(F.lit(None).cast(DoubleType())))
    # join reporter country names
    .join(m49_df.select(
              F.col("m49_code").alias("reporter_m49_key"),
              F.col("iso2_code").alias("reporter_iso2"),
              F.col("country_name").alias("reporter_name")
          ),
          F.col("reporter_m49") == F.col("reporter_m49_key"),
          "left"
    ).drop("reporter_m49_key")
    # join partner country names
    .join(m49_df.select(
              F.col("m49_code").alias("partner_m49_key"),
              F.col("iso2_code").alias("partner_iso2"),
              F.col("country_name").alias("partner_name")
          ),
          F.col("partner_m49") == F.col("partner_m49_key"),
          "left"
    ).drop("partner_m49_key")
    .drop("_source_file", "_ingested_at", "_bronze_layer")
    .select(
        "year",
        "reporter_m49", "reporter_iso2", "reporter_name",
        "flow_code", "flow_label",
        "partner_m49", "partner_iso2", "partner_name",
        "quantity_units", "net_weight_kg",
        "fob_value_usd", "primary_value_usd", "usd_per_unit"
    )
)

write_silver(comtrade, "comtrade_hs8701_exports",
             "UN Comtrade HS8701 trade flows with country names and unit values")

# Quick summary: top exporters by FOB value
print("\nTop exporters (all years) by total FOB value:")
spark.table("silver.comtrade_hs8701_exports") \
    .filter(F.col("flow_code") == "X") \
    .groupBy("reporter_name") \
    .agg(
        F.round(F.sum("fob_value_usd") / 1e9, 2).alias("total_fob_bn_usd"),
        F.sum("quantity_units").alias("total_units")
    ) \
    .orderBy(F.desc("total_fob_bn_usd")) \
    .show(15, truncate=False)

## Cell 10 — `silver.fred_annual_indices`

**Source:** Five bronze FRED tables  
**Transformations:**
- Aggregate all monthly series to annual averages
- Join into one wide table keyed on `year`
- Compute YoY change for each index
- Coverage: 2000–2025 (filter out pre-2000 historical data for tractors context)

| Column | FRED Series | Description |
|---|---|---|
| `avg_farm_equip_ppi` | WPU114 | Farm & Garden Machinery PPI (monthly → annual avg) |
| `avg_farm_equip_ppi_detail` | WPUFD41312 | Farm Equipment PPI – Final Demand |
| `avg_corn_ppi` | WPU012202 | Corn Producer Price Index |
| `avg_wheat_ppi` | WPU012101 | Wheat Producer Price Index |
| `net_farm_income_bn_usd` | A368RC1A027NBEA | US Net Farm Income (annual, $bn) |

In [ ]:
# ── Helper: aggregate monthly FRED series to annual average ─────────────────
def fred_annual_avg(table: str, value_col: str, alias: str):
    """Read a bronze FRED monthly table and return annual average DataFrame."""
    return (
        spark.table(f"{BRONZE_SCHEMA}.{table}")
        .withColumn("year", F.year("observation_date"))
        .filter(F.col("year") >= 2000)
        .groupBy("year")
        .agg(F.round(F.avg(value_col), 2).alias(alias))
    )

# ── Build individual annual series DataFrames ────────────────────────────────
df_equip_ppi   = fred_annual_avg("fred_farm_equipment_ppi",    "WPU114",        "avg_farm_equip_ppi")
df_equip_det   = fred_annual_avg("fred_farm_equip_ppi_detail", "WPUFD41312",    "avg_farm_equip_ppi_detail")
df_corn        = fred_annual_avg("fred_corn_price_index",       "WPU012202",     "avg_corn_ppi")
df_wheat       = fred_annual_avg("fred_wheat_price_index",      "WPU012101",     "avg_wheat_ppi")

# Farm income is already annual — just filter and rename
df_farm_income = (
    spark.table(f"{BRONZE_SCHEMA}.fred_us_net_farm_income")
    .withColumn("year", F.year("observation_date"))
    .filter(F.col("year") >= 2000)
    .select(
        F.col("year"),
        F.round(F.col("A368RC1A027NBEA"), 2).alias("net_farm_income_bn_usd")
    )
)

# ── Join all series on year ──────────────────────────────────────────────────
# Start with a year spine (2000-2025) to ensure no year is dropped in the join
year_spine = spark.range(2000, 2026).toDF("year").withColumn("year", F.col("year").cast(IntegerType()))

fred_wide = (
    year_spine
    .join(df_equip_ppi,   "year", "left")
    .join(df_equip_det,   "year", "left")
    .join(df_corn,        "year", "left")
    .join(df_wheat,       "year", "left")
    .join(df_farm_income, "year", "left")
)

# ── Add YoY change columns for each index ───────────────────────────────────
w_yr = Window.orderBy("year")

for col_name in ["avg_farm_equip_ppi", "avg_corn_ppi", "avg_wheat_ppi", "net_farm_income_bn_usd"]:
    yoy_col = f"{col_name}_yoy_pct"
    prev_col = f"_prev_{col_name}"
    fred_wide = (
        fred_wide
        .withColumn(prev_col, F.lag(col_name, 1).over(w_yr))
        .withColumn(yoy_col,
                    F.round(
                        (F.col(col_name) - F.col(prev_col)) / F.col(prev_col) * 100,
                        2
                    ))
        .drop(prev_col)
    )

write_silver(fred_wide, "fred_annual_indices",
             "Unified annual FRED indices: equipment PPI, crop prices, farm income")

spark.table("silver.fred_annual_indices") \
    .select(
        "year", "avg_farm_equip_ppi", "avg_corn_ppi", "avg_wheat_ppi",
        "net_farm_income_bn_usd",
        "avg_farm_equip_ppi_yoy_pct", "net_farm_income_bn_usd_yoy_pct"
    ) \
    .orderBy("year") \
    .show(26, truncate=False)

## Cell 11 — `silver.faostat_arable_land`

**Source:** `bronze.faostat_arable_land`  
**Transformations:**
- Filter to key countries: India, China, USA, Germany, Japan, Brazil, France, UK
- Rename `area` → `country_name` and standardise to canonical names
- Convert `value` (1000 ha) → `arable_land_mha` (million hectares) for readability
- Filter years ≥ 1990, keep only the `Arable land` element
- Drop FAOSTAT internal code columns not needed downstream

In [ ]:
raw_fao = spark.table(f"{BRONZE_SCHEMA}.faostat_arable_land")

# Countries of interest (using FAOSTAT area names from the raw data)
KEY_COUNTRIES_FAO = [
    "India", "China", "United States of America",
    "Germany", "Japan", "Brazil", "France",
    "United Kingdom", "Argentina", "Australia",
    "Russian Federation", "Turkey", "Pakistan", "Bangladesh",
]

# Canonical name override map (FAOSTAT name → display name)
fao_name_map = {
    "United States of America": "United States",
    "Russian Federation":       "Russia",
}
fao_map_expr = F.create_map([F.lit(v) for pair in fao_name_map.items() for v in pair])

faostat_silver = (
    raw_fao
    # keep only arable land element and relevant years
    .filter(
        (F.lower(F.col("element")) == "area") &    # 'Area' = the arable land area measure
        (F.col("year") >= 1990) &
        (F.col("area").isin(KEY_COUNTRIES_FAO)) &
        F.col("value").isNotNull()
    )
    # standardise country name
    .withColumn("country_name",
                F.coalesce(fao_map_expr[F.col("area")], F.col("area")))
    # convert 1000 ha → million ha
    .withColumn("arable_land_mha",
                F.round(F.col("value") / 1000.0, 3))
    .drop("_source_file", "_ingested_at", "_bronze_layer")
    .select(
        "country_name", F.col("area_code_m49").alias("m49_code"),
        "year",
        F.col("value").alias("arable_land_1000_ha"),
        "arable_land_mha",
        "unit", "flag"
    )
    .orderBy("country_name", "year")
)

write_silver(faostat_silver, "faostat_arable_land",
             "FAOSTAT arable land – key countries 1990+, million hectares")

spark.table("silver.faostat_arable_land") \
    .filter(F.col("year") >= 2010) \
    .orderBy("country_name", "year") \
    .show(30, truncate=False)

## Cell 12 — `silver.global_tractor_sales` (Unified Cross-Country View)

This is the **centrepiece** Silver table.
It merges tractor retail/sales data from India, the US, and global production
into a single unified schema:

| Column | Description |
|---|---|
| `year` | Calendar year |
| `country` | Country (canonical display name) |
| `country_code` | ISO-2 code |
| `data_type` | `retail_sales` / `production_domestic` |
| `hp_category` | `<40HP` / `40-100HP` / `100+HP` / `all` |
| `units_sold` | Units (retail or domestic production estimate) |
| `source` | Source attribution |

**Approach:**
1. **India**: annual fiscal-year data → calendar year; HP category = `all` (TMA does not break by HP)
2. **US**: AEM data is already by HP category; use retail units
3. **Other countries** (China, Germany, Japan): from `silver.global_tractor_production` domestic_sales_estimate; HP category = `all`

In [ ]:
# ── Source 1: India (annual, all HP) ────────────────────────────────────────
india_unified = (
    spark.table("silver.india_tractor_sales")
    .select(
        F.col("calendar_year").alias("year"),
        F.lit("India").alias("country"),
        F.lit("IN").alias("country_code"),
        F.lit("retail_sales").alias("data_type"),
        F.lit("all").alias("hp_category"),
        F.col("total_units_sold").alias("units_sold"),
        F.lit("TMA (Tractor Manufacturers Association of India)").alias("source")
    )
)

# ── Source 2: US by HP category ─────────────────────────────────────────────
us_raw = spark.table("silver.us_aem_tractor_sales_hp")

us_under40 = us_raw.select(
    F.col("year"),
    F.lit("United States").alias("country"),
    F.lit("US").alias("country_code"),
    F.lit("retail_sales").alias("data_type"),
    F.lit("<40HP").alias("hp_category"),
    F.col("under_40hp_units").alias("units_sold"),
    F.lit("AEM Farm Equipment Retail Statistics").alias("source")
)

us_mid = us_raw.select(
    F.col("year"),
    F.lit("United States").alias("country"),
    F.lit("US").alias("country_code"),
    F.lit("retail_sales").alias("data_type"),
    F.lit("40-100HP").alias("hp_category"),
    F.col("hp40_to_100_units").alias("units_sold"),
    F.lit("AEM Farm Equipment Retail Statistics").alias("source")
)

us_over100 = us_raw.select(
    F.col("year"),
    F.lit("United States").alias("country"),
    F.lit("US").alias("country_code"),
    F.lit("retail_sales").alias("data_type"),
    F.lit("100+HP").alias("hp_category"),
    F.col("over_100hp_units").alias("units_sold"),
    F.lit("AEM Farm Equipment Retail Statistics").alias("source")
)

# Also keep a US 'all' total row for easy year-total comparisons
us_total = us_raw.select(
    F.col("year"),
    F.lit("United States").alias("country"),
    F.lit("US").alias("country_code"),
    F.lit("retail_sales").alias("data_type"),
    F.lit("all").alias("hp_category"),
    F.col("total_units").alias("units_sold"),
    F.lit("AEM Farm Equipment Retail Statistics").alias("source")
)

# ── Source 3: Other countries from global production domestic estimates ───────
other_countries = (
    spark.table("silver.global_tractor_production")
    .filter(~F.col("country_code").isin("IN", "US"))   # exclude India & US (covered above)
    .select(
        F.col("year"),
        F.col("country_name").alias("country"),
        F.col("country_code"),
        F.lit("production_domestic").alias("data_type"),
        F.lit("all").alias("hp_category"),
        F.col("domestic_sales_estimate").alias("units_sold"),
        F.lit("TMA / CTA / JAMA / VDMA (compiled)").alias("source")
    )
)

# ── Union all sources ────────────────────────────────────────────────────────
global_sales = (
    india_unified
    .union(us_under40)
    .union(us_mid)
    .union(us_over100)
    .union(us_total)
    .union(other_countries)
    .orderBy("year", "country", "hp_category")
)

write_silver(global_sales, "global_tractor_sales",
             "Unified global sales: India + US (by HP) + China/DE/JP/other")

print("\nRecord count by country:")
spark.table("silver.global_tractor_sales") \
    .groupBy("country", "data_type") \
    .agg(F.count("*").alias("rows"),
         F.min("year").alias("min_year"),
         F.max("year").alias("max_year")) \
    .orderBy("country") \
    .show(20, truncate=False)

print("\nGlobal totals (all-category rows only) — top years:")
spark.table("silver.global_tractor_sales") \
    .filter(F.col("hp_category") == "all") \
    .groupBy("year") \
    .agg(F.sum("units_sold").alias("total_units")) \
    .orderBy(F.desc("year")) \
    .show(10)

## Cell 13 — Data Quality Checks

Perform automated quality assertions on all silver tables before promoting to Gold.
Checks:
1. Row counts ≥ expected minimums
2. No nulls in key columns
3. Numeric range checks (units_sold > 0, share percentages 0–100)
4. India seasonality sums to ~100 %
5. Brand shares sum to ~100 %

In [ ]:
from pyspark.sql.utils import AnalysisException

PASS  = "PASS "
FAIL  = "FAIL "
WARN  = "WARN "

results = []

def check(name: str, passed: bool, warn_only: bool = False, detail: str = ""):
    status = PASS if passed else (WARN if warn_only else FAIL)
    results.append((status, name, detail))

# ── 1. Row count minimums ────────────────────────────────────────────────────
min_rows = {
    "silver.india_tractor_sales":       20,
    "silver.india_tractor_seasonality": 12,
    "silver.india_tractor_brands":        5,
    "silver.us_aem_tractor_sales_hp":    10,
    "silver.global_tractor_production":  15,
    "silver.comtrade_hs8701_exports":   100,
    "silver.fred_annual_indices":        20,
    "silver.faostat_arable_land":        50,
    "silver.global_tractor_sales":       50,
}

for tbl, min_r in min_rows.items():
    try:
        n = spark.table(tbl).count()
        check(f"row_count: {tbl}", n >= min_r, detail=f"{n} rows (min {min_r})")
    except AnalysisException as e:
        check(f"row_count: {tbl}", False, detail=str(e))

# ── 2. No nulls in key columns ───────────────────────────────────────────────
null_checks = {
    "silver.india_tractor_sales":    ["calendar_year", "total_units_sold"],
    "silver.us_aem_tractor_sales_hp": ["year", "total_units"],
    "silver.global_tractor_sales":   ["year", "country", "units_sold"],
    "silver.fred_annual_indices":    ["year", "avg_farm_equip_ppi"],
}

for tbl, cols in null_checks.items():
    try:
        df = spark.table(tbl)
        for c in cols:
            null_n = df.filter(F.col(c).isNull()).count()
            check(f"no_nulls: {tbl}.{c}", null_n == 0, detail=f"{null_n} nulls")
    except AnalysisException as e:
        check(f"no_nulls: {tbl}", False, detail=str(e))

# ── 3. Numeric range checks ──────────────────────────────────────────────────
try:
    neg_sales = spark.table("silver.global_tractor_sales") \
        .filter(F.col("units_sold") <= 0).count()
    check("positive_units: silver.global_tractor_sales", neg_sales == 0,
          detail=f"{neg_sales} rows with units ≤ 0")
except AnalysisException as e:
    check("positive_units: silver.global_tractor_sales", False, detail=str(e))

try:
    bad_share = spark.table("silver.us_aem_tractor_sales_hp") \
        .filter(
            (F.col("under_40hp_share_pct") < 0) | (F.col("under_40hp_share_pct") > 100)
        ).count()
    check("share_pct_range: silver.us_aem_tractor_sales_hp", bad_share == 0,
          detail=f"{bad_share} rows out of [0,100]")
except AnalysisException as e:
    check("share_pct_range", False, detail=str(e))

# ── 4. India seasonality sums to ~100 % ──────────────────────────────────────
try:
    total_pct = spark.table("silver.india_tractor_seasonality") \
        .agg(F.sum("avg_share_pct")).collect()[0][0]
    check("seasonality_sum_100", abs(total_pct - 100) < 0.5,
          detail=f"sum = {total_pct:.2f}%")
except AnalysisException as e:
    check("seasonality_sum_100", False, detail=str(e))

# ── 5. Brand shares sum to ~100 % ────────────────────────────────────────────
try:
    brand_pct = spark.table("silver.india_tractor_brands") \
        .agg(F.sum("market_share_2023_pct")).collect()[0][0]
    check("brand_share_sum_100", abs(brand_pct - 100) < 1.0,
          detail=f"sum = {brand_pct:.2f}%")
except AnalysisException as e:
    check("brand_share_sum_100", False, detail=str(e))

# ── Print results ─────────────────────────────────────────────────────────────
print(f"\n{'STATUS':<6}  {'CHECK':<60}  DETAIL")
print("-" * 100)
fail_count = 0
for status, name, detail in results:
    print(f"{status}  {name:<60}  {detail}")
    if status == FAIL:
        fail_count += 1

print("-" * 100)
print(f"\nTotal checks: {len(results)}  |  Failures: {fail_count}")
if fail_count > 0:
    raise Exception(f"{fail_count} DQ check(s) failed — review output above before running Gold layer.")
else:
    print("All DQ checks passed. Ready for Notebook 3 — Gold Layer.")

## Cell 14 — Silver Layer Summary

Print a final inventory of all silver tables with row counts and column counts.

In [ ]:
SILVER_TABLES = [
    ("silver.india_tractor_sales",       "India annual sales — fiscal year, growth metrics"),
    ("silver.india_tractor_seasonality", "Monthly demand share — Kharif/Rabi season labels"),
    ("silver.india_tractor_brands",      "Brand market share — ranked, HP parsed"),
    ("silver.us_aem_tractor_sales_hp",   "US retail sales — HP segment %, YoY growth"),
    ("silver.global_tractor_production", "Global production — export intensity, domestic share"),
    ("silver.comtrade_hs8701_exports",   "UN Comtrade HS8701 — country names, USD/unit"),
    ("silver.fred_annual_indices",       "FRED annual indices — equipment PPI, crop prices, farm income"),
    ("silver.faostat_arable_land",       "FAOSTAT arable land — key countries, million ha"),
    ("silver.global_tractor_sales",      "Unified global sales — India + US (HP) + China/DE/JP"),
]

print(f"\n{'TABLE':<45}  {'ROWS':>8}  {'COLS':>5}  DESCRIPTION")
print("-" * 110)

for table, desc in SILVER_TABLES:
    try:
        df    = spark.table(table)
        rows  = df.count()
        cols  = len(df.columns)
        print(f"{table:<45}  {rows:>8,}  {cols:>5}  {desc}")
    except Exception as e:
        print(f"{table:<45}  {'ERROR':>8}         {e}")

print("-" * 110)
print("\nSilver layer complete.")
print("Next step → 03_gold_kpis.ipynb")

## Cell 15 — Next Steps

Silver layer is complete. All tables follow the unified schema and have passed DQ checks.

Proceed to **`03_gold_kpis.ipynb`** — Build analytical aggregates:

| Gold Table | Source Silver Table(s) | Description |
|---|---|---|
| `gold.market_summary` | `global_tractor_sales` | Global volume by year, top-5 countries, CAGR |
| `gold.india_seasonality` | `india_tractor_sales` + `india_tractor_seasonality` | Monthly demand estimates with season flags |
| `gold.hp_segment_trend` | `us_aem_tractor_sales_hp` | US HP category mix over time |
| `gold.price_vs_sales` | `fred_annual_indices` + `global_tractor_sales` | PPI / farm income vs. sales correlation |
| `gold.top_exporters` | `comtrade_hs8701_exports` | Top-10 countries by FOB value 2015–2022 |
| `gold.india_brand_share` | `india_tractor_brands` | Brand share ranking for dashboards |